# Creating a gold-standard dataset from pf8 and GenRE MEKONG data

The aim of this notebook is to create a gold-standard dataset for malaria genomics pipelines that predict drug resistance.

The following resources will be used:
1. [MalariaGEN Pf8](https://www.malariagen.net/data_package/open-dataset-plasmodium-falciparum-v80/)
2. [GenRE Mekong](https://www.malariagen.net/resource/29/)

## Rationale for selection of resources
Pipelines are usually tailored to a particular sequencing  technology, such as whole-genome or amplicon sequencing. While Pf8 is based on (selective) Whole Genome Sequencing (sWGS) data, GenRE Mekong utilised the SPotMalaria panel for amplicon sequencing (download available at [MalariaGEN](https://www.malariagen.net/wp-content/uploads/2023/11/20200705-GenRe-04b-SpotMalaria-SupplementaryFile1.xlsx)).
Some GenRE Mekong samples have been re-analysed with sWGS for Pf8, thus providing data for the same sample from two different technologies and two independent resistance phenotype calls.
We will use the intersect between Pf8 and GenRE Mekong as the gold-standard dataset.

In Pf8, drug-resistance phenotypes are inferred based on criteria described in [this document](https://pf8-release.cog.sanger.ac.uk/Pf8_resistance_classification.pdf)

## Set up environment
All packages imported here must be installed locally

In [1]:
import pandas as pd
import requests
import re
import os
import sys
from pathlib import Path
sys.path.append(str(Path("../../lib").resolve()))

from warnings import simplefilter 
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
pd.options.mode.copy_on_write = True

___custom ENA data helper module___
If this import fails, check that the ENA_data_helper.py script exists in the same directory as this Jupyter notebook. 

In [2]:
from ENA_data_helper import create_ena_data_frame_from_samples, align_ena_results_with_sample_data_genre_pf8

## Load Pf8 data
Obtain sample metadata and drug resistance inference data from Pf8.
Details can be found [here](https://www.malariagen.net/data_package/open-dataset-plasmodium-falciparum-v80/). Rename a few of the DR columns to be consistent with the GenRe names (this will be important later)

In [3]:
pf8_infer_resistance_df=pd.read_csv("https://pf8-release.cog.sanger.ac.uk/Pf8_inferred_resistance_status_classification.tsv", sep="\t")
pf8_infer_resistance_df.rename(
    columns = {
        'SP (uncomplicated)' : 'S-P',
        'SP (IPTp)' : 'S-P-IPTp'
    }, 
    inplace=True
)



In [4]:
pf8_samples_df=pd.read_csv("https://pf8-release.cog.sanger.ac.uk/metadata/Pf8_samples.txt", sep="\t")

## Merge Pf8 resistance and sample metadata
Rename the sample column with lower case for consistency across data; also rename two of the drug resistance columns to be consistent with the naming in GenRe (this will be important later)

In [5]:
pf8_samples_df.rename(
    columns = {
        'Sample' : 'sample',
    }, 
    inplace=True
)

In [6]:
pf8_df = pd.merge(pf8_infer_resistance_df, pf8_samples_df, on=["sample"], how="outer")
pf8_df

,sample,Chloroquine,Pyrimethamine,Sulfadoxine,Mefloquine,Artemisinin,Piperaquine,S-P,S-P-IPTp,AS-MQ,...,Admin level 1 longitude,Year,ENA,All samples same case,Population,% callable,QC pass,Exclusion reason,Sample type,Sample was in Pf7
0,FP0008-C,Undetermined,Undetermined,Undetermined,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,-9.832345,2014.0,ERR1081237,FP0008-C,AF-W,82.48,True,Analysis_set,gDNA,True
1,FP0009-C,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,...,-9.832345,2014.0,ERR1081238,FP0009-C,AF-W,88.95,True,Analysis_set,gDNA,True
2,FP0010-CW,Undetermined,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,...,-9.832345,2014.0,ERR2889621,FP0010-CW,AF-W,87.01,True,Analysis_set,sWGA,True
3,FP0011-CW,Undetermined,Resistant,Undetermined,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,...,-9.832345,2014.0,ERR2889624,FP0011-CW,AF-W,86.95,True,Analysis_set,sWGA,True
4,FP0012-CW,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,...,-9.832345,2014.0,ERR2889627,FP0012-CW,AF-W,89.86,True,Analysis_set,sWGA,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33320,SPT92268,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.618846,2000.0,ERR10940681,SPT92268,AF-W,0.10,False,Low_coverage,sWGA,False
33321,SPT92269,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.618846,2000.0,ERR11009733,SPT92269,AF-W,0.12,False,Low_coverage,sWGA,False
33322,SPT92270,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.618846,2000.0,ERR11009737,SPT92270,AF-W,0.01,False,Low_coverage,sWGA,False
33323,SPT94772,Resistant,Resistant,Resistant,Undetermined,Undetermined,Undetermined,Resistant,Sensitive,Undetermined,...,-16.401559,2017.0,ERR10789456,SPT94772,AF-W,64.96,True,Analysis_set,sWGA,False


## Load GenRe Mekong data
We use [this Excel document](https://github.com/GenRe-Mekong/Data/blob/main/Pf-GenRe-GRCv1.4-Full-PublicRelease-20240514.xlsx) from the GenRe Mekong Github project, release v1.4 14/05/24 (latest release)  

Sample IDs for samples that were re-analysed in Pf8 are the same in both projects, i.e. it is possible to merge GenRe Mekong and Pf8 data on sample ID.  

For more information on the data in this spreadsheet, consult the [GenRe Mekong data dictionary](https://github.com/GenRe-Mekong/Documents/blob/main/GRC-DataDictionary/Pf-GenRe-GRCv1.4-DataDictionary.xlsx) and [this publication](https://rdcu.be/eUOoe).

In [7]:
GenRe_df = pd.read_excel(
    "https://raw.githubusercontent.com/GenRe-Mekong/Data/main/Pf-GenRe-GRCv1.4-Full-PublicRelease-20240514.xlsx",
    sheet_name="GRC"
)

# make sample ID column name match Pf8
GenRe_df.rename(columns={'SampleId': 'sample'}, inplace=True)

## Merge GenRe Mekong and Pf8 data
Sample IDs for samples that were analysed in Pf8 are not changed, i.e. we can merge GenRe and Pf8 on sample ID.  
Retain only rows that have a sample ID that is found in GenRe and Pf8.

In [8]:
genre_pf8_merged_df = pd.merge(GenRe_df, pf8_df, on=["sample"], suffixes=('_GRMK','_pf8'), how="inner") 

## Remove GenRe Agena data
In the GenRe Mekong project, three techniques were used:
- Agena
- AmpSeqV1
- AmpSeqV2

These are recorded in the "Process" column of the GenRe Mekong data. Only AmpSeq data is relevant here because we are building a dataset for genotyping by sequencing. "Agena" refers to ["Agena Bioscience MassARRAY"](https://link.springer.com/protocol/10.1007/978-1-4939-6442-0_5), a technique based on PCR amplification of SNP regions that does not use sequencing and should thus not be used in this dataset.  

Count samples by process used in GenRe Mekong and "Sample type" in Pf8.  The Pf8 sample type, according to the [Pf8 data release README](https://pf8-release.cog.sanger.ac.uk/Pf8_README.txt), refers to
"Amplification technology used on the sample (MDA, gDNA or sWGA)", where "gDNA" means non-selective WGS, as opposed to selective sWGS. 

In [9]:
genre_pf8_merged_df.groupby(['Process','Sample type']).size().to_frame('count').reset_index()

,Process,Sample type,count
0,Agena,sWGA,3690
1,AmpSeqV1,gDNA,64
2,AmpSeqV1,sWGA,558
3,AmpSeqV2,sWGA,2730


-> 3690 samples in GenRe Mekong are based on Agena array, not on sequencing.  
Remove those samples.

In [10]:
genre_pf8_merged_seq_df = genre_pf8_merged_df[ genre_pf8_merged_df['Process'] != "Agena" ]

In [11]:
genre_pf8_merged_seq_df.groupby(['Process','Sample type']).size().to_frame('count').reset_index()

,Process,Sample type,count
0,AmpSeqV1,gDNA,64
1,AmpSeqV1,sWGA,558
2,AmpSeqV2,sWGA,2730


## Identify concordant drug-resistance calls and high-quality data
Both projects, GenRe Mekong and Pf8, provide drug-resistance phenotype predictions.  the final gold-standard dataset should contain only samples that have the same phenotype predictions in both projects and are of high quality.  


Identify columns that are found in both GenRe and Pf8

In [12]:
list(GenRe_df.columns.intersection( pf8_df.columns).values)

['sample',
 'Study',
 'Year',
 'Country',
 'Artemisinin',
 'Piperaquine',
 'Mefloquine',
 'Chloroquine',
 'Pyrimethamine',
 'Sulfadoxine',
 'DHA-PPQ',
 'AS-MQ',
 'S-P',
 'S-P-IPTp']

-> all except 'sample', 'Study', 'Year' and 'Country'are drug-resistance phenotypes.  
Create a list of field names to compare.

In [13]:
phenotypes = [
 'Artemisinin',
 'Piperaquine',
 'Mefloquine',
 'Chloroquine',
 'Pyrimethamine',
 'Sulfadoxine',
 'DHA-PPQ',
 'AS-MQ',
 'S-P',
 'S-P-IPTp']

Generate a concordance/discordance summary table

In [14]:
summary_rows = []

for p in phenotypes:
    
    pf8_col = f"{p}_pf8"
    genre_col = f"{p}_GRMK"
    
    counts = (
        genre_pf8_merged_seq_df
            .groupby([pf8_col, genre_col])
            .size()
            .unstack(fill_value=0)
            .stack()
    )
    
    # Flatten index into column names
    counts.index = [f"{pf8}/{genre}" for pf8, genre in counts.index]
    
    row = counts.to_dict()
    row["drug"] = p
    
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).fillna(0)
summary_df = summary_df.set_index("drug").reset_index()

summary_df


,drug,Resistant/Missing,Resistant/Resistant,Resistant/Sensitive,Resistant/Undetermined,Sensitive/Missing,Sensitive/Resistant,Sensitive/Sensitive,Sensitive/Undetermined,Undetermined/Missing,Undetermined/Resistant,Undetermined/Sensitive,Undetermined/Undetermined,Resistant/SensitiveWithMissingness,Sensitive/SensitiveWithMissingness,Undetermined/SensitiveWithMissingness
0,Artemisinin,14,1542,1,30.0,36,0,917,13.0,153,118,81,61.0,0.0,0.0,0.0
1,Piperaquine,56,964,1,0.0,521,36,581,0.0,439,276,92,0.0,0.0,0.0,0.0
2,Mefloquine,0,15,1,0.0,11,4,1484,0.0,3,6,189,0.0,0.0,0.0,0.0
3,Chloroquine,5,2728,0,1.0,0,0,185,1.0,1,15,4,26.0,0.0,0.0,0.0
4,Pyrimethamine,28,2784,0,0.0,4,0,77,4.0,15,40,3,11.0,0.0,0.0,0.0
5,Sulfadoxine,34,2207,0,3.0,7,0,651,3.0,18,22,5,16.0,0.0,0.0,0.0
6,DHA-PPQ,55,910,1,8.0,167,30,623,3.0,308,222,62,35.0,0.0,496.0,46.0
7,AS-MQ,0,15,1,0.0,5,4,1503,0.0,3,6,152,0.0,0.0,479.0,32.0
8,S-P,29,2250,0,2.0,10,0,544,2.0,20,48,7,33.0,0.0,21.0,0.0
9,S-P-IPTp,6,90,0,4.0,15,0,1162,8.0,76,16,8,1526.0,0.0,54.0,1.0


The data shows that there are discordant as well as concordant calls for all drug resistance phenotypes, thus it is important to filter for samples that are concordant. At this stage, we conservatively only consider "Sensitive" and "Resistant" phenotypes, and consider GRMK "SensitiveWithMissingness" as "Sensitive" 

In [15]:
genre_pf8_conc_df = genre_pf8_merged_seq_df
for p in phenotypes:
    cols = [ p + '_GRMK', p + '_pf8' ]
    genre_pf8_conc_df = genre_pf8_conc_df[
        ( (((genre_pf8_conc_df[cols[0]] == "Sensitive") | (genre_pf8_conc_df[cols[0]] == "SensitiveWithMissingness")) & (genre_pf8_conc_df[cols[1]] == "Sensitive")) |
          ((genre_pf8_conc_df[cols[0]] == "Resistant") & (genre_pf8_conc_df[cols[1]] == "Resistant")) |
          ((genre_pf8_conc_df[cols[0]] == "Undetermined") & (genre_pf8_conc_df[cols[1]] == "Undetermined")) 
        )
    ]
len(genre_pf8_conc_df)

1037

Use the Pf8 published quality metric to further filter out samples that were deemed to be low-quality in Whole-genome sequencing

In [16]:
genre_pf8_conc_df = genre_pf8_conc_df[ 
    (genre_pf8_conc_df['% callable'] >= 85 ) &
    (genre_pf8_conc_df['QC pass'] == True )
]
len(genre_pf8_conc_df)

965

Reduce the dataset to a smaller number of relevant columns. 
For country of origin, use the Pf8 value and rename the column accordingly.  
Column 'ampseq_process' refers to GenRe Mekong data only.

In [17]:
keep_cols = [
    'sample', 'Country_pf8', 'Process', 'Artemisinin_GRMK', 'Piperaquine_GRMK',
    'Mefloquine_GRMK', 'Chloroquine_GRMK', 'Pyrimethamine_GRMK',
    'Sulfadoxine_GRMK', 'DHA-PPQ_GRMK', 'AS-MQ_GRMK', 'S-P_GRMK', 'S-P-IPTp_GRMK',
    'Chloroquine_pf8', 'Pyrimethamine_pf8',
    'Sulfadoxine_pf8', 'Mefloquine_pf8', 'Artemisinin_pf8',
    'Piperaquine_pf8', 'AS-MQ_pf8',
    'DHA-PPQ_pf8', 'S-P_pf8', 'S-P-IPTp_pf8'
]

genre_pf8_conc_s_df = genre_pf8_conc_df[keep_cols]
genre_pf8_conc_s_df.rename(columns={'Country_pf8': 'country', 'Process': 'ampseq_process'}, inplace=True)

Combine the phenotype calls into single column each (they are all concordant due to the filter used)

In [18]:
for p in phenotypes:
    # just to be safe
    if not genre_pf8_conc_s_df[p + '_GRMK'].equals(genre_pf8_conc_s_df[p + '_pf8']):
        raise Exception('phenotypes are not concordant but they should be if filtered correctly')
    genre_pf8_conc_s_df = genre_pf8_conc_s_df.rename(columns={p + '_GRMK': p}, errors="raise")
    genre_pf8_conc_s_df = genre_pf8_conc_s_df.drop([ p + '_pf8' ], axis=1)

Overview of final data

In [19]:
genre_pf8_conc_s_df

,sample,country,ampseq_process,Artemisinin,Piperaquine,Mefloquine,Chloroquine,Pyrimethamine,Sulfadoxine,DHA-PPQ,AS-MQ,S-P,S-P-IPTp
3730,RCN12025,Vietnam,AmpSeqV1,Resistant,Sensitive,Sensitive,Resistant,Resistant,Resistant,Sensitive,Sensitive,Resistant,Undetermined
3731,RCN12026,Vietnam,AmpSeqV1,Resistant,Resistant,Sensitive,Resistant,Resistant,Resistant,Resistant,Sensitive,Resistant,Undetermined
3733,RCN12028,Vietnam,AmpSeqV1,Resistant,Resistant,Sensitive,Resistant,Resistant,Resistant,Resistant,Sensitive,Resistant,Undetermined
3734,RCN12031,Vietnam,AmpSeqV1,Resistant,Resistant,Sensitive,Resistant,Resistant,Resistant,Resistant,Sensitive,Resistant,Undetermined
3735,RCN12032,Vietnam,AmpSeqV1,Resistant,Sensitive,Sensitive,Resistant,Resistant,Resistant,Sensitive,Sensitive,Resistant,Sensitive
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6975,RCN26767,Laos,AmpSeqV2,Resistant,Sensitive,Sensitive,Resistant,Resistant,Resistant,Sensitive,Sensitive,Resistant,Sensitive
6976,RCN26775,Laos,AmpSeqV2,Sensitive,Sensitive,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive
6987,RCN26820,Laos,AmpSeqV2,Resistant,Sensitive,Sensitive,Resistant,Resistant,Resistant,Sensitive,Sensitive,Resistant,Sensitive
7021,RCN26932,Laos,AmpSeqV2,Resistant,Sensitive,Sensitive,Resistant,Resistant,Resistant,Sensitive,Sensitive,Resistant,Sensitive


## Find INSDC run accessions for the data 
Using functions from the custom ENA data helper tool (file ENA_data_helper.py in the working directory), perform a search on ENA by sample ID and retrieve data for run accessions. For samples sequenced by both Pf8 and GenRe,  we expect 4 runs for each sample: one for Pf8 (WGS) and three for GenRe-Mekong.  

First, we read in a pre-prepared file that maps ENA run accessions for the GenRe Mekong data to sub-panel names (this [notebook](genre_mekong_acc_map/get_genre_run_to_panel_map.ipynb) explains why this is necessary). The resulting lookup table will be used multiple times later on.

In [20]:
genre_run_to_panel_df = pd.read_csv("genre_mekong_accmap/genre_ena_runacc_to_panel_map.csv")
panel_dict = genre_run_to_panel_df.set_index("run_accession")["panel"].to_dict()

We next call a function that performs a search for ENA data by sample IDs in sample data frame

In [21]:
genre_pf8_conc_s_ena_result = create_ena_data_frame_from_samples(genre_pf8_conc_s_df)

We finally call a second function that aligns the sample dataframe with the ENA dataframe, such that only samples that have complete data in ENA for both projects are retained. 

In [22]:
genre_pf8_conc_s_ena_df, ena_result_filtered = align_ena_results_with_sample_data_genre_pf8(sample_data=genre_pf8_conc_s_df, ena_result=genre_pf8_conc_s_ena_result, genre_panel_map=panel_dict)
genre_pf8_conc_s_ena_df
ena_result_filtered

,run_accession,experiment_accession,study_accession,sample,read_count,center_name,library_strategy,sample_accession,library_name
0,ERR14388658,ERX13789777,PRJEB85801,RCN12043,74846,The GenRe-Mekong Project;GenRe-Mekong,AMPLICON,SAMEA117703772,RCN12043_GRC2
1,ERR14388702,ERX13789821,PRJEB85801,RCN12060,75290,The GenRe-Mekong Project;GenRe-Mekong,AMPLICON,SAMEA117703787,RCN12060_GRC1
2,ERR14388728,ERX13789847,PRJEB85801,RCN12070,908,The GenRe-Mekong Project;GenRe-Mekong,AMPLICON,SAMEA117703795,RCN12070_SPEC
3,ERR14388730,ERX13789849,PRJEB85801,RCN12071,65022,The GenRe-Mekong Project;GenRe-Mekong,AMPLICON,SAMEA117703796,RCN12071_GRC2
4,ERR14388838,ERX13789957,PRJEB85801,RCN12112,56576,The GenRe-Mekong Project;GenRe-Mekong,AMPLICON,SAMEA117703832,RCN12112_GRC2
...,...,...,...,...,...,...,...,...,...
55,ERR14397547,ERX13798666,PRJEB85801,RCN26767,86724,The GenRe-Mekong Project;GenRe-Mekong,AMPLICON,SAMEA117706735,RCN26767_GRC2
56,ERR14397548,ERX13798667,PRJEB85801,RCN26767,1532,The GenRe-Mekong Project;GenRe-Mekong,AMPLICON,SAMEA117706735,RCN26767_SPEC
57,ERR14397990,ERX13799109,PRJEB85801,RCN26938,122312,The GenRe-Mekong Project;GenRe-Mekong,AMPLICON,SAMEA117706883,RCN26938_GRC1
58,ERR14397991,ERX13799110,PRJEB85801,RCN26938,109946,The GenRe-Mekong Project;GenRe-Mekong,AMPLICON,SAMEA117706883,RCN26938_GRC2


### Save results: concordant phenotype data
In summary: this table of 904 rows represents samples that (a) have been sequenced by amplicon sequencing for GenRe Mekong and WGS for Pf8, (b) are not deemed to be low-quality from the WGS assay (using coverage/depth metrics), (c) have fully concordant phenotype calls (inferred from genotypes) between Pf8 and GenRe Mekong, and (b) are identifiable at (and retrievable from) ENA for both WGS and AmpSeq.   

We save the final dataset to three files:

- ```../Pf8-GenReMekong_concordant_phenotypes.csv``` - contains the resistance status for the selected samples (one row for each sample)
- ```../Pf8-GenReMekong_concordant_phenotypes.wgs.INDSC_manifest.wgs.csv``` - contains the collected Pf8 (WGS) ENA run accessions for the samples  
- ```../Pf8-GenReMekong_concordant_phenotypes.INDSC_manifest.spotmalaria.csv``` - contains the collected GenRe Mekong (SpotMalaria amplicon) ENA run accessions for the samples  

We also keep a full version of the samples file (with the complete set of columns) for reference:
- ```Pf8-GenReMekong_concordant_phenotypes_allcols.csv```

In [23]:
genre_pf8_conc_s_ena_df.to_csv('Pf8-GenReMekong_concordant_phenotypes_allcols.csv', index=False)

final_keep_cols=['sample', 'country', 'ampseq_process',
                 'Artemisinin', 'Piperaquine', 'Mefloquine', 'Chloroquine', 'Pyrimethamine', 'Sulfadoxine','DHA-PPQ', 'AS-MQ','S-P', 'S-P-IPTp'
                 ]
genre_pf8_conc_s_ena_df[final_keep_cols].to_csv('../Pf8-GenReMekong_concordant_phenotypes.csv', index=False)

ena_result_filtered_wgs = ena_result_filtered[ena_result_filtered["library_name"].str.endswith("_WGS")]
ena_result_filtered_spot = ena_result_filtered[ena_result_filtered["library_name"].str.endswith(("_GRC1", "_GRC2", "_SPEC"))]

ena_keep_cols=['sample','sample_accession','library_name','experiment_accession','run_accession']
ena_result_filtered_wgs[ena_keep_cols].sort_values(by=["sample","library_name"]).to_csv('../Pf8-GenReMekong_concordant_phenotypes.INSDC_manifest.wgs.csv', index=False)
ena_result_filtered_spot[ena_keep_cols].sort_values(by=["sample","library_name"]).to_csv('../Pf8-GenReMekong_concordant_phenotypes.INSDC_manifest.spotmalaria.csv', index=False)

***

## Add genotypes
In the above steps, concordance was defined on high-level drug resistance phenotype calls.  
Adding the underlying genotype calls will provide a lower-level truth dataset that can be used to validate genotype calls of a novel pipeline.  
___Note___ that "genotype" calls are provided at amino acid level - not nucelotide level - in both studies. 

### Genes and haplotypes
For the purpose of this gold-standard dataset, the intention is to identify samples that are fully concordant on their haplotype calls for important genes between Pf8 and GenRe Mekong, i.e. two different sequencing techinques and two pipelines have resulted in the same haplotype call.  

This dataset will focus on 8 key drug resistance genes where drug resistance is determined by non-synonymous point mutations:
- dhfr
- crt
- dhps
- kelch13
- mdr1
- mdr2
- fd
- arps10

This list of genes and "core" amino acid positions was compiled according to Table SM1 (Markers associated with drug resistance for P. falciparum) in the [SpotMalaria Platform Technical Notes and Methods](https://ngs.sanger.ac.uk/production/malaria/Resource/29/20200705-GenRe-04a-SpotMalaria-0.39.pdf). For Pf8, details about drug resistance classification can be found in in [this document](https://pf8-release.cog.sanger.ac.uk/Pf8_resistance_classification.pdf).  

This results in the following list of drug-resistance haplotypes to be considered for the gold- standard dataset:

| Gene | Amino Acid position |
|---|---|
| dhfr | 51, 59, 108, 164 |
| crt | 72, ~~73~~\*, 74, 75, 76, 326, 356 |
| dhps | 436, 437, 540, 581, 613 |
| kelch13 | 349-726 |
| mdr1 | 86, 184, 1246 |
| mdr2 | 484 |
| fd | 193 |
| arps10 | 127, 128 |
\* Position crt:73 is listed as a core position in the reference table but it is not provided in GenRe Mekong data and it is always WT in Pf8 data, hence this position will be ignored in the following code.  

For more details on gene haplotypes in P. falciparum see the [Pf-HaploAtlas](https://www.malariagen.net/article/introducing-pf-haploatlas-a-new-app-to-track-malaria-parasite-mutations/). 

NOTE: this set does not cover the complete set of genetic markers used to infer drug resistance in these projects. Some resistance markers have more complex genotypes (e.g. resistance to Piperaquine as marked by an increased copy number of the Plasmepsin gene). For this reason, the set of samples that we would derive by looking for concordance in these simple (point mutation) markers will not be a strict subset or superset of the samples derived by phenotype concordance above; some samples may be concordant at phenotype level but discordant at genotype level; similarly, some samples may be concordant at (simple) genotype level but discordant at phenotype level (when considering the wider set of genotype-derived phenotypes). For ease of understanding of these datasets, we define the following sample set as the strict subset of the phenotype-concordant samples that have concordant point-mutation genotypes between Pf8 and GRMK.  
       

### Load Pf8 genotype calls from data release

In [24]:
pf8_gt_df = pd.read_csv("https://pf8-release.cog.sanger.ac.uk/Pf8_drug_resistance_marker_genotypes.tsv", sep="\t", keep_default_na=False)

### Merge Pf8 genotypes 
Merge into into existing dataframe  ```genre_pf8_merged_seq_df```  created above, which is the merged data for Pf8 and GenRe Mekong, with array data removed. 

In [25]:
pf8_gt_df.rename(columns={'Sample': 'sample'}, inplace=True)

genre_pf8_merged_seq_gt_df = pd.merge(
    genre_pf8_merged_seq_df, 
    pf8_gt_df, 
    on=["sample"], 
    suffixes=('_GRMK','_pf8'), 
    how="inner") 

Reduce the dataset to a smaller number of relevant columns. 
For country of origin, use the Pf8 value and rename the column accordingly.  
Column 'ampseq_process' refers to GenRe Mekong data only.

In [26]:
drop_cols=['Source', 'Country_pf8','Study_GRMK', 'Year_GRMK', 'Month',
       'TimePoint', 'AdmDiv1', 'AdmDiv1_GID', 'AdmDiv2',
       'AdmDiv2_GID', 'Artemisinin_GRMK', 'Piperaquine_GRMK',
       'Mefloquine_GRMK', 'Chloroquine_GRMK', 'Pyrimethamine_GRMK',
       'Sulfadoxine_GRMK', 'DHA-PPQ_GRMK', 'AS-MQ_GRMK', 'S-P_GRMK',
       'S-P-IPTp_GRMK', 'Species', 'GenBarcode', 'GenBarcodeMissing',
       'GenBarcodeHet', 'COI', 'pm23-break', 'pm23-qPCR', 'mdr1-qPCR', 'species-aSeq',
       'species-qPCR', 'species-barcode', 'mdr1_dup_call', 'pm2_dup_call',
       'Chloroquine_pf8', 'Pyrimethamine_pf8',
       'Sulfadoxine_pf8', 'Mefloquine_pf8', 'Artemisinin_pf8',
       'Piperaquine_pf8', 'AS-MQ_pf8', 'S-P_pf8','S-P-IPTp_pf8',
       'DHA-PPQ_pf8', 'Study_pf8', 'Admin level 1',
       'Country latitude', 'Country longitude', 'Admin level 1 latitude',
       'Admin level 1 longitude', 'Year_pf8', 'ENA',
       'All samples same case', 'Population', 
       'Exclusion reason', 'Sample type', 'Sample was in Pf7']

genre_pf8_merged_seq_gt_df.drop(columns=drop_cols, inplace=True)
genre_pf8_merged_seq_gt_df.rename(columns={'Country_GRMK': 'country', 'Process': 'ampseq_process'}, inplace=True)

The dataframe now contains Pf8 and GenRe Mekong genotypes for all samples that have been sequenced (excluding Array data) in both projects and can be linked via sample ID.

### Create new dataframe with amino acid genotype calls
Create a list of gene/aa-positions from the above table and use it to compile a new data frame with amino acid calls from Pf8 and GenRe Mekong for these core positions. The kelch13 list needs to be built separately because the data is provided in separate column in GenRe Mekong and as a single column of comma-separated lists in Pf8.


The new columns will have a unified naming scheme:  
"core_mutation_{GENE}_{AAPos}_\[pf8/GMRK\]"

The majority of genotypes can be extracted from the Pf8 and GenRe data by a simple pattern.  
The respective pattern of genotype calls by gene and AA position is as follows:  
__Pf8__: {GENE}\_{AAPos} (example "crt_75\[N\]")  
__GenRe Mekong__: Pf{GENE}:{AAPos} (example: "PfCRT:75")  

Create a set of new column names and respective Pf8 and GenRe column names accordingly.  
There are ___two excpetions___: 
1. kelch13: in both datasets, kelch13 data is provided as a column of lists of AA positions that need to be converted separately
2. arps10: provided as AA pos 127-128 in Pf8 and just pos 127 in GenRe. Needs to be converted separately


In [27]:
genes_pos = {
    'dhfr': [51, 59, 108, 164],
    'crt': [72, 74, 75, 76, 326, 356], # omitting pos 73, see above for explanation
    'dhps': [436, 437, 540, 581, 613],
    'mdr1': [86, 184, 1246],
    'mdr2': [484],
    'fd': [193]
}

gt_cols = {}
for gene, pos_list in genes_pos.items():
    gt_cols[ gene ] = {}
    for pos in pos_list:
        gt_cols[ gene ][ pos ] = {
            'pf8_pattern': gene + '_' + str(pos) + '\[[A-Z]\]',
            'GRMK_name': 'Pf' + gene.upper() + ':' + str(pos),
            'new_col_basename': 'core_mutation_' + gene + '_' + str(pos)
        }

### Add new genotype columns 
Add new genotype columns for the core mutation positions to dataframe  ```genre_pf8_merged_seq_gt_df```

In [28]:
for gene in gt_cols:
    for pos in gt_cols[ gene ]:
        
        # genotype from Pf8
        pf8_pattern = gt_cols[gene][pos]['pf8_pattern']
        new_col_pf8 = gt_cols[gene][pos]['new_col_basename'] + '_pf8'
        pf8_col = [ c for c in genre_pf8_merged_seq_gt_df if re.match(pf8_pattern, c) ]
        if not pf8_col:
            raise ValueError(f'could not find Pf8 column starting with "{pf8_pattern}"')
        elif len(pf8_col) > 1:
            raise ValueError(f'found more than one Pf8 column starting with "{pf8_pattern}":{", ".join(pf8_col)}')
        else:
            genre_pf8_merged_seq_gt_df[new_col_pf8] = genre_pf8_merged_seq_gt_df[pf8_col[0]].copy()
            
        # genotype from GenRe
        genre_pattern = gt_cols[gene][pos]['GRMK_name']
        new_col_genre = gt_cols[gene][pos]['new_col_basename'] + '_GRMK'
        if genre_pattern in genre_pf8_merged_seq_gt_df:
            genre_pf8_merged_seq_gt_df[new_col_genre] = genre_pf8_merged_seq_gt_df[genre_pattern].copy()
        else:
            raise ValueError(f'could not find GenRe column "{genre_pattern}"')
                                                                         

Add columns for arps10: use only position arsp10:127, which is provided as a column inf GenRe and needs to be extracted from column 'arps10_127-128\[VD\]' in Pf8.

In [29]:
arps10_basename = 'core_mutation_arps10_127'
genre_pf8_merged_seq_gt_df[ arps10_basename + '_GRMK'] = genre_pf8_merged_seq_gt_df['PfARPS10:127'].copy()
genre_pf8_merged_seq_gt_df[ arps10_basename + '_pf8'] = genre_pf8_merged_seq_gt_df['arps10_127-128[VD]'].astype(str).str[0]

### Add columns for kelch13
Add columns for all kelch13 AA positions for which data exists in at least one sample in both projects.  
__Pf8__:  
    _column_: 'kelch13_349-726_ns_changes'  
    _content_: comma separated list of haplotypes at Kelch13 positions 349-726. Each haplotype contains one or more non-synonymous variations, separateed by "/". Homozygous mutations are shown in upper case and heterozygous in lower case. Extract from [here (heading "Pf8_drug_resistance_marker_genotypes.tsv"](https://pf8-release.cog.sanger.ac.uk/Pf8_README.txt):

*Explanation of amino acid columns in crt, dhfr and dhps:*

*Each value can have a single haplotype if homozygous or two haplotypes separated by a comma if heterozygous. It is possible to have heterozygous calls where both amino acid haplotypes are the same. The heterozygosity here is at the nucleotide level. These could perhaps be considered homozygous alt.*

*"\-" represents missing (missing genotype in at least one of the positions)*

*"\*" represents an unphased het followed by another het. Because hets are unphased it is not possible to resolve the two haplotypes. These are perhaps best considered missing.*

*"\!" represents a frame-shift in the haplotype. These are perhaps best considered missing.*

*Explanation of non-synonymous changes (ns_changes): Non-synonymous mutations are shown in the form: \<REF\>\<POS\>\<ALT\>. Homozygous mutations are shown in upper case and heterozygous in lower case. The nomenclature for amino acids described above is also used in this field.*
    
__GenRe__:  
    _column_: 'Pfkelch13'  
    _content_: Comma-separated list of amino acid mutations. ___NOTE___ this differs from Pf8. In GenRe, the comma delimits mutations in the same haplotype, while in Pf8 (see above) a comma delimts haplotypes. There is no disctiontion between homo- and heterozygous mutations. Extract the [GenRe Mekong user guide](https://github.com/GenRe-Mekong/Documents/blob/b918eae72b2834537955db97b552ca2596239483/GRC_UserGuide-v1.4.pdf):

- *“WT” indicates that no nonsynonymous mutation is detected in the relevant domains.*
- *An amino acid mutation encoded using “XnnnY” notation, where X is the wild-type amino acid, Y is the mutated amino acid, and nnn is the amino acid position in the protein sequence. For example, “C580Y” indicates a C→Y mutation at position 580 in the kelch13 gene.*
- *A comma-separated list of one or more of the above genotypes, in cases where more than one mutation was identified. This is likely to happen in multiclonal infections for example “WT,C580Y” would be reported for a sample that contains parasites with the C580Y mutation, as well as wild-type parasites.*
- *A dash (“‐”) indicates a missing genotype, i.e., because a large portion of the kelch13 gene could not be genotyped for this sample.*
- *“\<NA\>” indicates that the sample was not tested for kelch13 mutations.*

Show the unique contents of the respective kelch13 columns in Pf8 and GenRe Mekong: 

In [30]:
genre_pf8_merged_seq_gt_df['kelch13_349-726_ns_changes'].unique()

array(['C580Y', '', 'c580y,p553l*', '-', 'c580y,v445i/c580y', 'c580y',
       'R561H', 'F446I', 'c580y,r561h/c580y', '!*', 'c580y,a578d/c580y',
       'c580y,p419s/c580y', 'P553L', 'R539T', 'Y493H', 'e668k', 'g625e',
       'g718s', 'e362k', 'A578S', 'e567k', '!', 'd353n', 'd399n', 'r515k',
       'c580y,t573i/c580y', 'p570l', 'c580y,f506y/c580y', 'p615l',
       'G453S', 'd584n', 'd397n', 'c580y,p574l/c580y',
       'e401k/c580y,c580y', 'G357D', 'C580Y/E705K', 'G449D', 'r404k',
       'c580y,a564t/c580y', 'c580y,c580y/q654h', 'c580y,c580y/i723v',
       'c580y,c580y/p615t', 'c580y,y456f/c580y', 't474i',
       'c580y,d512n/c580y', 'r539t', 'r539t,r539t/c580f'], dtype=object)

In [31]:
genre_pf8_merged_seq_gt_df['Pfkelch13'].unique()

array(['C580Y', 'WT,A432S,C580Y', 'WT', 'WT,P553L,C580Y', '-', 'R561H',
       'WT,C469.,C580Y', 'WT,P419S,C580Y', 'P553L', 'WT,C580Y', 'R539T',
       'Y493H', 'WT,G453S', 'WT,G545R', 'WT,D464N', 'WT,M351I', 'A578S',
       'WT,E567K', 'WT,F662Y', 'WT,G548V', 'WT,K658R', 'WT,D353N',
       'WT,D399N', 'WT,R515K', 'WT,E431K', 'WT,P615L', 'WT,G453S,W518.',
       'WT,D397N', 'WT,P574L,C580Y', 'WT,G538D,C580Y', 'WT,G358V,C580Y',
       'WT,C580Y,N599D', 'WT,C580Y,Q654R', 'WT,S549P,C580Y', 'WT,D373G',
       'WT,S485G,C580Y', 'WT,G496D,C580Y', 'WT,C580Y,R622K',
       'WT,I540V,C580Y', 'WT,C580Y,D648G', 'WT,C580Y,F656L',
       'WT,E401K,C580Y', 'WT,D547G', 'WT,F442S,C580Y', 'WT,R471S,C580Y',
       'WT,G357D,C580Y', 'WT,C542R', 'WT,L444S,C580Y', 'WT,C580Y,G595D',
       'WT,I376M,C580Y', 'WT,C580Y,E612.', 'WT,W565.,C580Y',
       'WT,H384R,C580Y', 'WT,G449D,C580Y', 'WT,A557V', 'WT,R404K',
       'WT,C542Y,C580Y', 'WT,C580Y,S649P', 'WT,F434S,C580Y',
       'WT,Y493H,S550P,C580Y', 'WT,K378

### Create individual kelch13 mutation columns
Create a column with a name ```core_mutation_kelch13_{pf8/GRMK}```. Values will a string list comprising elements of the format ```{WT allele}{Pos}{Mutant allele}```

In [32]:
# Pf8
def standardise_pf8_muts( muts ):
    muts = str(muts)
    if muts:
        if muts == "-" or muts == "!" or muts == "*" or muts == "!*":
            # missing
            muts = "Missing"
        else:
            if re.search(r'[1-9]', muts):
                muts = muts.replace('/',',') # treat all mutations individually, regardless of haplotype grouping
                muts = muts.replace('*','')
                muts = muts.upper()
                mut_list = list(set(muts.split(",")))

                muts = ",".join(sorted(mut_list, key=lambda x: int(re.search(r"\d+", x).group())))
    else:
        muts = "WT"
    return muts
            
genre_pf8_merged_seq_gt_df["core_mutation_kelch13_pf8"] = genre_pf8_merged_seq_gt_df["kelch13_349-726_ns_changes"].map( standardise_pf8_muts )
genre_pf8_merged_seq_gt_df["core_mutation_kelch13_pf8"].unique()


array(['C580Y', 'WT', 'P553L,C580Y', 'Missing', 'V445I,C580Y', 'R561H',
       'F446I', 'R561H,C580Y', 'A578D,C580Y', 'P419S,C580Y', 'P553L',
       'R539T', 'Y493H', 'E668K', 'G625E', 'G718S', 'E362K', 'A578S',
       'E567K', 'D353N', 'D399N', 'R515K', 'T573I,C580Y', 'P570L',
       'F506Y,C580Y', 'P615L', 'G453S', 'D584N', 'D397N', 'P574L,C580Y',
       'E401K,C580Y', 'G357D', 'C580Y,E705K', 'G449D', 'R404K',
       'A564T,C580Y', 'C580Y,Q654H', 'C580Y,I723V', 'C580Y,P615T',
       'Y456F,C580Y', 'T474I', 'D512N,C580Y', 'R539T,C580F'], dtype=object)

In [33]:
# GenRe Mekong
def standardise_genre_muts( muts ):
    muts = str(muts)
    if muts:
        if muts == "-":
            muts = "Missing"
        elif muts == "WT":
            muts = "WT"
        elif re.search(r'[1-9]', muts):
            mut_list = muts.split(',')
            mut_list = [x for x in mut_list if x != 'WT']
            muts = ",".join( sorted(mut_list, key=lambda x: int(re.search(r"\d+", x).group())))
    else:
        muts = "Missing"
    return muts

genre_pf8_merged_seq_gt_df["core_mutation_kelch13_GRMK"] = genre_pf8_merged_seq_gt_df["Pfkelch13"].map( standardise_genre_muts )
genre_pf8_merged_seq_gt_df["core_mutation_kelch13_GRMK"].unique()


array(['C580Y', 'A432S,C580Y', 'WT', 'P553L,C580Y', 'Missing', 'R561H',
       'C469.,C580Y', 'P419S,C580Y', 'P553L', 'R539T', 'Y493H', 'G453S',
       'G545R', 'D464N', 'M351I', 'A578S', 'E567K', 'F662Y', 'G548V',
       'K658R', 'D353N', 'D399N', 'R515K', 'E431K', 'P615L',
       'G453S,W518.', 'D397N', 'P574L,C580Y', 'G538D,C580Y',
       'G358V,C580Y', 'C580Y,N599D', 'C580Y,Q654R', 'S549P,C580Y',
       'D373G', 'S485G,C580Y', 'G496D,C580Y', 'C580Y,R622K',
       'I540V,C580Y', 'C580Y,D648G', 'C580Y,F656L', 'E401K,C580Y',
       'D547G', 'F442S,C580Y', 'R471S,C580Y', 'G357D,C580Y', 'C542R',
       'L444S,C580Y', 'C580Y,G595D', 'I376M,C580Y', 'C580Y,E612.',
       'W565.,C580Y', 'H384R,C580Y', 'G449D,C580Y', 'A557V', 'R404K',
       'C542Y,C580Y', 'C580Y,S649P', 'F434S,C580Y', 'Y493H,S550P,C580Y',
       'K378E,C580Y', 'A481P', 'C580Y,Q654H', 'Y456F,C580Y',
       'T363K,C580Y', 'W470R', 'Y502H,C580Y', 'S364P', 'C580Y,R597I',
       'C580Y,S600F', 'N609D', 'T350A,L488F', 'V566I', 'T

In [34]:

pf8c = genre_pf8_merged_seq_gt_df["core_mutation_kelch13_pf8"].value_counts()
genrec = genre_pf8_merged_seq_gt_df["core_mutation_kelch13_GRMK"].value_counts()

counts = pd.DataFrame({
    "pf8_count": pf8c,
    "genre_count": genrec
}).fillna(0).astype(int).reset_index()

counts = counts.rename(columns={"index": "value"})
counts = counts.sort_values("genre_count", ascending=False)
counts

,value,pf8_count,genre_count
10,C580Y,1520,1604
96,WT,965,998
65,Missing,367,203
78,R539T,71,74
81,R561H,5,3
...,...,...,...
79,"R539T,C580F",1,0
82,"R561H,C580Y",1,0
92,"V445I,C580Y",1,0
89,T474I,1,0


### Identify concordant samples
Filter for samples that are fully concordant on all core mutation genotype calls, i.e. all columns added with the "core_mutation_" prefix above.  

In [35]:
def is_concordant(row, *core_mut_col_basenames):
    """
    Return True if all 'core_mutation' columns match between 
    Pf8 and GenRe Mekong
    """
    for col in core_mut_col_basenames:
        if row[col + '_pf8'] != row[col + '_GRMK'] or row[col + '_pf8'] == "Missing" or row[col + '_GRMK'] == "Missing":
            return False
    return True

regex = re.compile(r'^(core_mutation_.+)_(pf8|GRMK)$')
core_mut_col_basenames = set(
    [ regex.sub(r'\1', x) for x in genre_pf8_merged_seq_gt_df.columns.values if regex.match(x) ]
)
genre_pf8_merged_seq_gt_df = genre_pf8_merged_seq_gt_df.copy()
genre_pf8_merged_seq_gt_df['fully_concordant_gt'] = genre_pf8_merged_seq_gt_df.apply(is_concordant, axis=1, args=core_mut_col_basenames)


Count fully concordant (on all genotypes reported) vs. not fully concordant samples across the two projects:

In [36]:
genre_pf8_merged_seq_gt_df.groupby( ['fully_concordant_gt']).size().to_frame('count').reset_index()

,fully_concordant_gt,count
0,False,917
1,True,2049


Approximately 2/3rds of samples are fully concordant on all drug-resistance "core mutation" genotypes between Pf8 (WGS) and GenRe Mekong (AmpSeq).

### Create a filtered dataset containing only fully concordant samples

In [37]:
genre_pf8_merged_seq_gt_concord_df = genre_pf8_merged_seq_gt_df[
    (genre_pf8_merged_seq_gt_df['fully_concordant_gt'] == True ) 
]

In [38]:
genre_pf8_merged_seq_gt_concord_df

,sample,ampseq_process,country,Pfkelch13,PfCRT,PfDHFR,PfDHPS,PfMDR1,pm23-Amp,mdr1-Amp,...,core_mutation_mdr1_1246_GRMK,core_mutation_mdr2_484_pf8,core_mutation_mdr2_484_GRMK,core_mutation_fd_193_pf8,core_mutation_fd_193_GRMK,core_mutation_arps10_127_GRMK,core_mutation_arps10_127_pf8,core_mutation_kelch13_pf8,core_mutation_kelch13_GRMK,fully_concordant_gt
0,RCN01775,AmpSeqV2,VN,C580Y,CVIET,IRNL,SGNGA,NFD,Amplified,NaN,...,D,I,I,Y,Y,M,M,C580Y,C580Y,True
1,RCN01776,AmpSeqV2,VN,C580Y,CVIET,IRNL,SGNGA,NFD,Amplified,NaN,...,D,I,I,Y,Y,M,M,C580Y,C580Y,True
2,RCN01777,AmpSeqV2,VN,C580Y,CVIET,IRNL,SGNGA,NFD,Amplified,NaN,...,D,I,I,Y,Y,M,M,C580Y,C580Y,True
3,RCN01778,AmpSeqV2,VN,C580Y,CVIET,IRNI,AGEAA,NYD,-,NaN,...,D,I,I,Y,Y,M,M,C580Y,C580Y,True
4,RCN01779,AmpSeqV2,VN,C580Y,CVIET,IRNL,SGNGA,NFD,-,NaN,...,D,I,I,Y,Y,M,M,C580Y,C580Y,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2959,RCN26945,AmpSeqV2,LA,R539T,CVIET,IRNI,AGEAA,NFD,WT,WT,...,D,I,I,D,D,M,M,R539T,R539T,True
2960,RCN26946,AmpSeqV2,LA,C580Y,CVIET,IRNL,SGNGA,NFD,WT,WT,...,D,I,I,Y,Y,M,M,C580Y,C580Y,True
2962,RCN26952,AmpSeqV2,LA,R539T,CVIET,IRNI,AGEAA,NFD,WT,WT,...,D,I,I,D,D,M,M,R539T,R539T,True
2963,RCN26953,AmpSeqV2,LA,R539T,CVIET,IRNI,AGEAA,NFD,WT,WT,...,D,I,I,D,D,M,M,R539T,R539T,True


There are 2049 high-quality samples where all reported drug-resitance genotype positions are fully concordant between Pf8 and GenRe Mekong. However, as explained above, this will include samples that are not strictly phenotype-concordant (using the wider set of phenotypes). For simplicity of understanding the dataset as whole, we remove samples that are not phenotype concordant (according to our list above)

In [39]:
genre_pf8_merged_seq_gt_concord_df = genre_pf8_merged_seq_gt_concord_df[genre_pf8_merged_seq_gt_concord_df['sample'].isin(genre_pf8_conc_s_ena_df['sample'])]
len(genre_pf8_merged_seq_gt_concord_df)

836

Keep only relevant columns. As all samples are fully concordant, keep only one column per core mutation and drop the pf8/GRMK suffix.

In [40]:
keep_cols = ['sample', 'country', 'ampseq_process', 'fully_concordant_gt']

core_mut_cols = [x for x in genre_pf8_merged_seq_gt_concord_df.columns.values if x.startswith('core_mutation')]
keep_cols.extend( core_mut_cols )
genre_pf8_merged_seq_gt_concord_small_df = genre_pf8_merged_seq_gt_concord_df[keep_cols]
genre_pf8_merged_seq_gt_concord_small_df.rename({'Country_GRMK':'Country'})

regex = re.compile(r'^(core_mutation_.+)_(pf8|GRMK)$')
for col in core_mut_cols:
    col_basename = regex.sub(r'\1', col)
    genre_pf8_merged_seq_gt_concord_small_df[col_basename] = genre_pf8_merged_seq_gt_concord_small_df[col].copy()
    genre_pf8_merged_seq_gt_concord_small_df.drop(col, axis=1, inplace=True)

## Find sample run accessions

As with the previous dataset, use functions from the custom ENA data helper tool (file ENA_data_helper.py), to perform a search on ENA by sample ID and retrieve data for run accessions. 

In [41]:
genre_pf8_merged_seq_gt_concord_small_ena_result = create_ena_data_frame_from_samples(genre_pf8_merged_seq_gt_concord_small_df)

genre_pf8_merged_seq_gt_concord_small_ena_df, genre_pf8_merged_seq_gt_concord_small_ena_result_filtered  = align_ena_results_with_sample_data_genre_pf8(
    sample_data=genre_pf8_merged_seq_gt_concord_small_df, ena_result=genre_pf8_merged_seq_gt_concord_small_ena_result, genre_panel_map=panel_dict
)

### Save file of fully concordant samples
As before, save the results to four files:
- ```../Pf8-GenReMekong_concordant_genotypes.csv``` - Samples where all drug-resistance genotypes are identical in Pf8 (WGS) and GenRe Mekong (AmpSeq)
- ```../Pf8-GenReMekong_concordant_genotypes.INSDC_manifest.[wgs|spotmalaria].csv``` - ENA run accessions for above samples
- ```Pf8-GenReMekong_concordant_genotypes_allcols.csv``` - all columns of the samples dataframe, for reference

In [42]:
genre_pf8_merged_seq_gt_concord_small_ena_df.to_csv('Pf8-GenReMekong_concordant_genotypes_allcols.csv', index=False)

final_drop_cols=['fully_concordant_gt',
        'INSDC_Pf8_readcount','INSDC_GenRe_GRC1_readcount','INSDC_GenRe_GRC2_readcount','INSDC_GenRe_SPEC_readcount',
        'INSDC_Pf8','INSDC_GenRe_GRC1','INSDC_GenRe_GRC2','INSDC_GenRe_SPEC'
        ]

genre_pf8_merged_seq_gt_concord_small_ena_df.drop(columns=final_drop_cols).to_csv('../Pf8-GenReMekong_concordant_genotypes.csv', index=False)

ena_result_filtered_wgs = genre_pf8_merged_seq_gt_concord_small_ena_result_filtered[genre_pf8_merged_seq_gt_concord_small_ena_result_filtered["library_name"].str.endswith("_WGS")]
ena_result_filtered_spot = genre_pf8_merged_seq_gt_concord_small_ena_result_filtered[genre_pf8_merged_seq_gt_concord_small_ena_result_filtered["library_name"].str.endswith(("_GRC1", "_GRC2", "_SPEC"))]

ena_keep_cols=['sample','sample_accession','library_name','experiment_accession','run_accession']
ena_result_filtered_wgs[ena_keep_cols].sort_values(by=["sample","library_name"]).to_csv('../Pf8-GenReMekong_concordant_genotypes.INSDC_manifest.wgs.csv', index=False)
ena_result_filtered_spot[ena_keep_cols].sort_values(by=["sample","library_name"]).to_csv('../Pf8-GenReMekong_concordant_genotypes.INSDC_manifest.spotmalaria.csv', index=False)

***

## Distinct genotype patterns and representative samples
Final dataset is based on the fully concordant genotypes dataset, but will only contain one representative sample for every distinct pattern of haplotypes.   
This dataset is based on the previous dataset, which means only samples with downloadable FASTQ files are taken into consideration here.

First, get a count of different patterns.

In [43]:
core_mut_cols = [x for x in genre_pf8_merged_seq_gt_concord_small_ena_df.columns.values if x.startswith('core_mutation')]
genre_pf8_merged_seq_gt_concord_small_ena_df.groupby( core_mut_cols).size().to_frame('count').reset_index()

,core_mutation_dhfr_51,core_mutation_dhfr_59,core_mutation_dhfr_108,core_mutation_dhfr_164,core_mutation_crt_72,core_mutation_crt_74,core_mutation_crt_75,core_mutation_crt_76,core_mutation_crt_326,core_mutation_crt_356,...,core_mutation_dhps_581,core_mutation_dhps_613,core_mutation_mdr1_86,core_mutation_mdr1_184,core_mutation_mdr1_1246,core_mutation_mdr2_484,core_mutation_fd_193,core_mutation_arps10_127,core_mutation_kelch13,count
0,I,R,N,I,C,I,D,T,N,I,...,A,A,N,Y,D,I,D,V,WT,1
1,I,R,N,I,C,I,D,T,N,I,...,A,A,N,Y,D,I,Y,M,WT,2
2,I,R,N,I,C,I,D,T,N,I,...,A,"A,T",N,Y,D,I,Y,V,C580Y,1
3,I,R,N,I,C,I,D,T,N,I,...,A,T,N,F,D,I,Y,M,C580Y,6
4,I,R,N,I,C,I,D,T,N,I,...,A,A,N,Y,D,I,D,V,WT,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,N,R,N,I,C,I,E,T,N,I,...,A,A,N,Y,D,T,D,V,WT,1
70,N,R,N,I,C,M,N,K,N,I,...,A,A,N,Y,D,T,D,V,WT,3
71,N,R,N,I,C,M,N,K,N,I,...,A,A,N,Y,D,T,"D,Y",V,WT,1
72,N,R,N,I,C,"M,I","N,D","K,T",N,I,...,A,A,N,Y,D,T,D,V,WT,2


There are 74 distinct patterns of genotypes.
Extract a list of sample IDs of representative samples for each pattern.

In [44]:
genre_pf8_merged_seq_gt_concord_patterns_final_df = (
    genre_pf8_merged_seq_gt_concord_small_ena_df.sort_values('sample')
      .drop_duplicates(subset=core_mut_cols, keep='first')
)

genre_pf8_merged_seq_gt_concord_patterns_final_df

,sample,country,ampseq_process,fully_concordant_gt,core_mutation_dhfr_51,core_mutation_dhfr_59,core_mutation_dhfr_108,core_mutation_dhfr_164,core_mutation_crt_72,core_mutation_crt_74,...,core_mutation_arps10_127,core_mutation_kelch13,INSDC_Pf8,INSDC_Pf8_readcount,INSDC_GenRe_GRC1,INSDC_GenRe_GRC1_readcount,INSDC_GenRe_GRC2,INSDC_GenRe_GRC2_readcount,INSDC_GenRe_SPEC,INSDC_GenRe_SPEC_readcount
38,RCN12025,VN,AmpSeqV1,True,I,R,N,L,C,I,...,M,C580Y,ERR15625306,13349952.0,ERR14388603,32819.0,ERR14388604,29409.0,ERR14388605,1094.0
43,RCN12032,VN,AmpSeqV1,True,I,R,N,I,C,I,...,M,C580Y,ERR15625311,14982320.0,ERR14388624,34578.0,ERR14388625,29608.0,ERR14388626,960.0
44,RCN12033,VN,AmpSeqV1,True,I,R,N,I,C,I,...,M,C580Y,ERR15625312,11466623.0,ERR14388627,35936.0,ERR14388628,30872.0,ERR14388629,812.0
55,RCN12051,VN,AmpSeqV1,True,I,R,N,I,C,I,...,M,WT,ERR15625325,17759967.0,ERR14388678,33855.0,ERR14388679,29508.0,ERR14388680,891.0
60,RCN12060,VN,AmpSeqV1,True,N,R,N,I,C,I,...,V,WT,ERR15625332,11040932.0,ERR14388702,37645.0,ERR14388703,34137.0,ERR14388704,958.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2648,RCN25190,LA,AmpSeqV2,True,I,R,N,L,C,I,...,M,C580Y,ERR15628307,11291740.0,ERR14395863,29707.0,ERR14395864,26510.0,ERR14395865,1511.0
2711,RCN25319,LA,AmpSeqV2,True,I,R,N,I,C,I,...,V,WT,ERR15628436,13840592.0,ERR14396172,22311.0,ERR14396173,19174.0,ERR14396174,1850.0
2803,RCN25946,LA,AmpSeqV2,True,I,R,N,L,C,I,...,M,R539T,ERR15628559,12205062.0,ERR14396598,5481.0,ERR14396599,29780.0,ERR14396600,583.0
2907,RCN26663,LA,AmpSeqV2,True,I,R,N,I,C,I,...,V,WT,ERR15628710,18256799.0,ERR14397318,56912.0,ERR14397319,43277.0,ERR14397320,1032.0


### Save file
Sumary: this file contains 74 "representative samples", each represent a distinct pattern of drug-resistance haplotypes.  

As before, we save the data to 4 files:
- ```../Pf8-GenReMekong_concordant_genotypes_representative_samples.csv```  
- ```../Pf8-GenReMekong_concordant_genotypes_representative_samples.INSDC_manifest.[wgs|spotmalaria].csv```
- ```Pf8-GenReMekong_concordant_genotypes_representative_samples_allcols.csv```
 

In [45]:
genre_pf8_merged_seq_gt_concord_patterns_final_df.to_csv('Pf8-GenReMekong_concordant_genotypes_representative_samples_allcols.csv', index=False)

final_drop_cols=['fully_concordant_gt',
        'INSDC_Pf8_readcount','INSDC_GenRe_GRC1_readcount','INSDC_GenRe_GRC2_readcount','INSDC_GenRe_SPEC_readcount',
        'INSDC_Pf8','INSDC_GenRe_GRC1','INSDC_GenRe_GRC2','INSDC_GenRe_SPEC'
        ]

genre_pf8_merged_seq_gt_concord_patterns_final_df.drop(columns=final_drop_cols).to_csv('../Pf8-GenReMekong_concordant_genotypes_representative_samples.csv', index=True)

ena_concord_pattern_df = genre_pf8_merged_seq_gt_concord_small_ena_result_filtered[
    genre_pf8_merged_seq_gt_concord_small_ena_result_filtered["sample"].isin(genre_pf8_merged_seq_gt_concord_patterns_final_df["sample"])
]

ena_result_filtered_wgs = ena_concord_pattern_df[ena_concord_pattern_df["library_name"].str.endswith("_WGS")]
ena_result_filtered_spot = ena_concord_pattern_df[ena_concord_pattern_df["library_name"].str.endswith(("_GRC1", "_GRC2", "_SPEC"))]

ena_keep_cols=['sample','sample_accession','library_name','experiment_accession','run_accession']
ena_result_filtered_wgs[ena_keep_cols].sort_values(by=["sample","library_name"]).to_csv('../Pf8-GenReMekong_concordant_genotypes_representative_samples.INSDC_manifest.wgs.csv', index=False)
ena_result_filtered_spot[ena_keep_cols].sort_values(by=["sample","library_name"]).to_csv('../Pf8-GenReMekong_concordant_genotypes_representative_samples.INSDC_manifest.spotmalaria.csv', index=False)

***

## Summary of files created
In conclusion, the following files have been created in the __parent directory__ of this notebook: 

__Pf8-GenReMekong_concordant_phenotypes.csv__
A table of 904 samples that have been sequenced by amplicon sequencing for GenRe Mekong and WGS for Pf8 and the  sample is marked as high-quality in Pf8 and all drug-resistance phenotye calls (inferred from genotypes) are fully concordant between Pf8 and GenRe Mekong.  

__Pf8-GenReMekong_concordant_genotypes.csv__
A table of 836 samples (a subset of the above) concordant genotype calls in Pf8 and GenRe Mekong for all drug-resistance mutations that are based on SNPs. This is more suitable for pipelines that do not make drug-resistance phenotype calls.

__Pf8-GenReMekong_concordant_genotypes_representative_samples.csv__
A selection of 74 samples from the concordant genotype dataset, where each sample represents a unique pattern of drug-resistance SNPs. Due to the smaller size, this set may be preferable to the full set of 836 samples with concordant genotypes.

Each file is paired with corresponding INSDC manifests that can be used to download the datasets from ENA